In [1]:
import numpy as np
import scipy.io as scio
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
data, data_X, data_y = [],[],[]
file_path = "D:\\1 ارشد\پایان نامه\\گزارش 23 - ارائه پروپوزال\\TE Data\\"

for mode in ['1','2','3','4','5','6']:
    mode_name = 'm' + mode + 'd00'
    temp = scio.loadmat(file_path + mode_name)[mode_name]
    #data.append(temp.copy())
    data_X.append(temp[:,[0, 1, 2, 3, 4, 5, 8, 9, 10, 12, 13, 15, 17, 18, 20, 21]].copy())
    data_y.append(temp[:,32].copy())  #purge E

np.round(pd.DataFrame(data_X[1]),3).head()


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0.307,737.367,8033.449,8.561,31.701,46.110,124.205,0.354,90.306,2705.738,26.315,3328.012,65.418,4.898,108.524,91.509
1,0.306,731.896,8083.074,8.655,31.630,46.230,124.201,0.354,90.252,2705.751,26.224,3328.070,65.433,5.247,108.513,91.354
2,0.309,725.945,8033.404,8.579,31.647,46.043,124.195,0.353,90.237,2704.870,26.214,3327.154,65.427,4.299,108.524,91.433
3,0.309,751.991,8011.162,8.496,31.965,46.419,124.191,0.353,90.220,2705.372,26.371,3327.004,65.424,4.100,108.541,91.600
4,0.305,723.574,8042.783,8.588,31.571,46.129,124.201,0.354,90.245,2705.674,26.466,3327.606,65.413,5.565,108.532,91.634


In [7]:
data_ar = np.hstack([data_X[0][:7201,:], data_y[0][:7201].reshape(-1,1)])
data_ar[0]

array([2.71032874e-01, 3.64973941e+03, 4.45132079e+03, 9.22314238e+00,
       3.23928998e+01, 4.75597536e+01, 1.22898796e+02, 1.97826456e-01,
       9.19932673e+01, 2.70352107e+03, 2.52575586e+01, 3.33081033e+03,
       6.66283028e+01, 0.00000000e+00, 1.02480028e+02, 9.22625307e+01,
       1.61374012e+01])

In [8]:
index_permute = np.random.permutation(7200)

In [11]:
data_splitted = [data_ar[index_permute[:1440],:], data_ar[index_permute[1440:2*1440],:], data_ar[index_permute[2*1440:3*1440],:], data_ar[index_permute[3*1440:4*1440],:], data_ar[index_permute[4*1440:],:]]

fold = []

train1 = np.vstack([data_splitted[1], data_splitted[2], data_splitted[3], data_splitted[4]])
test1 = data_splitted[0]
fold.append([train1, test1])

train2 = np.vstack([data_splitted[0], data_splitted[2], data_splitted[3], data_splitted[4]])
test2 = data_splitted[1]
fold.append([train2, test2])

train3 = np.vstack([data_splitted[0], data_splitted[1], data_splitted[3], data_splitted[4]])
test3 = data_splitted[2]
fold.append([train3, test3])

train4 = np.vstack([data_splitted[0], data_splitted[1], data_splitted[2], data_splitted[4]])
test4 = data_splitted[3]
fold.append([train4, test4])

train5 = np.vstack([data_splitted[1], data_splitted[2], data_splitted[3], data_splitted[0]])
test5 = data_splitted[4]
fold.append([train5, test5])


In [12]:
from importlib import reload
import my_BLS_Regression_5
reload(my_BLS_Regression_5)
from my_BLS_Regression_5 import BLS_regression

In [13]:
fit_RMSE_sum, fit_MAPE_sum, pred_RMSE_sum, pred_MAPE_sum = [],[],[],[]

for traintest in fold:
    
    model = BLS_regression(0.001, 1, 10, 8, 6, 3, activation='relu6')
    model.fit(traintest[0][:,:16], traintest[0][:,16])
    model.predict(traintest[1][:,:16], traintest[1][:,16])
    fit_RMSE_sum.append(model.fit_RMSE)
    fit_MAPE_sum.append(model.fit_MAPE)
    pred_RMSE_sum.append(model.pred_RMSE)
    pred_MAPE_sum.append(model.pred_MAPE)

In [17]:
[i*100 for i in fit_MAPE_sum]

[1.2312145564311752,
 1.2266804495186192,
 1.2443398564091466,
 1.23279142573443,
 1.215908568241024]

In [21]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# ---------------------------
# تعریف تابع clip بر اساس پرسنتایل‌ها
# ---------------------------
def percentile_clip_bounds(X, lower_percentile=0.5, upper_percentile=99.5):
    """محاسبه مقادیر حد پایین و بالا برای clipping"""
    clip_min = np.percentile(X, lower_percentile, axis=0)
    clip_max = np.percentile(X, upper_percentile, axis=0)
    return clip_min, clip_max

# ---------------------------
# داده‌ها: فرض می‌کنیم X_train, X_test, y_train, y_test آماده‌اند
# X_train, X_test = داده‌های فیچر (16 فیچر)
# y_train, y_test = خروجی درصد مولی E
# ---------------------------

# ---------------------------
# مرحله ۱: StandardScaler و clip (همان preprocessing BLS)
# ---------------------------

fit_RMSE_sum_2, fit_MAPE_sum_2, pred_RMSE_sum_2, pred_MAPE_sum_2 = [],[],[],[]

for traintest in fold:
    scaler = StandardScaler()
    scaler.fit(traintest[0][:,:16])
    X_train_scaled = scaler.transform(traintest[0][:,:16])
    X_test_scaled = scaler.transform(traintest[1][:,:16])
    # Clip به پرسنتایل‌های 0.5 و 99.5
    clip_min, clip_max = percentile_clip_bounds(X_train_scaled)
    X_train_scaled = np.clip(X_train_scaled, clip_min, clip_max)
    X_test_scaled = np.clip(X_test_scaled, clip_min, clip_max)

    # ---------------------------
    # مرحله ۲: تعریف و آموزش درخت تصمیم (CART Regression Tree)
    # ---------------------------
    cart_tree = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=20,
    criterion='squared_error',
    random_state=0
    )
    cart_tree.fit(X_train_scaled, traintest[0][:,16])

    # ---------------------------
    # RMSE روی داده آموزش (fit RMSE)
    # ---------------------------
    y_train_pred = cart_tree.predict(X_train_scaled)
    rmse_fit = np.sqrt(mean_squared_error(traintest[0][:,16], y_train_pred))
    fit_RMSE_sum_2.append(rmse_fit)
    # ---------------------------
    # مرحله ۳: پیش‌بینی و ارزیابی RMSE
    # ---------------------------
    y_pred = cart_tree.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(traintest[1][:,16], y_pred))
    pred_RMSE_sum_2.append(rmse)




In [22]:
pred_RMSE_sum_2, fit_RMSE_sum_2

([0.2470088435832298,
  0.24961396792146118,
  0.2389665552780354,
  0.24324781337415277,
  0.25543309253412183],
 [0.2436734507281642,
  0.2432314153636029,
  0.24556647375756427,
  0.24238491184662433,
  0.23973847666948594])

In [23]:
pred_RMSE_sum, fit_RMSE_sum

([0.24618490630712453,
  0.250371009880324,
  0.23803670339251595,
  0.2463557700190744,
  0.2574035535679727],
 [0.24661040891204233,
  0.2455743005835263,
  0.24865390673244755,
  0.2466391834156424,
  0.24391234102356976])

In [30]:
# ---------------------------
# مرحله ۴: نمایش مسیرهای برگ‌ها (اختیاری برای تفسیرپذیری)
# ---------------------------
from sklearn.tree import export_text
tree_rules = export_text(cart_tree, feature_names=[f"X{i}" for i in range(16)])
print(tree_rules)


|--- X12 <= 0.85
|   |--- X12 <= -1.07
|   |   |--- X8 <= -2.25
|   |   |   |--- X7 <= 0.16
|   |   |   |   |--- value: [16.27]
|   |   |   |--- X7 >  0.16
|   |   |   |   |--- value: [16.44]
|   |   |--- X8 >  -2.25
|   |   |   |--- X7 <= -0.88
|   |   |   |   |--- value: [16.26]
|   |   |   |--- X7 >  -0.88
|   |   |   |   |--- value: [16.14]
|   |--- X12 >  -1.07
|   |   |--- X0 <= 1.51
|   |   |   |--- X0 <= 0.16
|   |   |   |   |--- value: [16.09]
|   |   |   |--- X0 >  0.16
|   |   |   |   |--- value: [16.12]
|   |   |--- X0 >  1.51
|   |   |   |--- X7 <= -0.26
|   |   |   |   |--- value: [15.94]
|   |   |   |--- X7 >  -0.26
|   |   |   |   |--- value: [16.09]
|--- X12 >  0.85
|   |--- X7 <= 0.82
|   |   |--- X12 <= 1.85
|   |   |   |--- X11 <= 0.17
|   |   |   |   |--- value: [16.09]
|   |   |   |--- X11 >  0.17
|   |   |   |   |--- value: [16.01]
|   |   |--- X12 >  1.85
|   |   |   |--- X0 <= 0.83
|   |   |   |   |--- value: [16.02]
|   |   |   |--- X0 >  0.83
|   |   |   |   

In [44]:
from sklearn.tree import export_graphviz
export_graphviz(cart_tree, out_file="tree.dot",
                feature_names=[f"X{i}" for i in range(16)],
                filled=True, rounded=True)
